In [ ]:
from generate_adaptive_seq_and_static_rec import *

In [ ]:
add_prob_to_score = False
truncate_continuous_distribution = True

In [ ]:
# =========================
# Script Parameters
# =========================
seed = 0
dataset = "ml-25m" # "ml-25m" / "netflix" / "amazon_books_2018_filtered_15cu_80ci"
first_stage_model_name = "CPMF"

num_users_to_sample = 10000

positive_threshold = 40   # keep all positives if user has at most positive_threshold positives
positive_cap = 40         # if user has more than positive_threshold positives, sample positive_cap

negative_sampling_mode = "relative"   # "fixed" or "relative"
num_negatives = 180                # total negatives per user
# The following parameters are used only if mode == "relative"
negatives_per_positive = 10
min_num_negatives = 100
max_num_negatives = 150

negative_kind = "mixed_es"         # "random", "hard_es", or "mixed_es"
hard_negative_fraction = 0.25       # fraction of negatives drawn from the hard pool
hard_pool_factor = 5               # hard pool size = factor * num_hard

max_size = None                    # max total candidate set size
selection_method = "pos_with_neg_sampling"

BASE_DIR = "./experiments"

In [4]:
# Set seed for reproducibility
random.seed(seed)
np.random.seed(seed)
rng = np.random.default_rng(seed)

In [5]:
scores_type = "scores_with_prob" if add_prob_to_score else "equal_scores"
fs_models_and_data_path = f'{FIRST_STAGE_MODELS_AND_DATA_MAIN_PATH}{dataset}/seed={seed}/'

data, first_stage_model = load_first_stage_model_and_data(fs_models_and_data_path,
                                                          first_stage_model_name)

/lv_local/home/coralscharf/anaconda3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data prepared: 162541 users, 32720 items.
19777302 train, 4866016 validation and 302552 test interactions.


In [ ]:
if negative_sampling_mode == "fixed":
    neg_desc = f"fixed{num_negatives}"
elif negative_sampling_mode == "relative":
    neg_desc = (
        f"rel{negatives_per_positive}-"
        f"min{min_num_negatives}-max{max_num_negatives}"
    )
else:
    raise ValueError(
        f"Unknown negative_sampling_mode: {negative_sampling_mode}"
    )

max_desc = "" if max_size is None else f"Max{max_size}-"

if negative_kind == "mixed_es":
    neg_kind_desc = f"mixedES-hf{hard_negative_fraction}-pf{hard_pool_factor}"
elif negative_kind == "hard_es":
    neg_kind_desc = f"hardES-pf{hard_pool_factor}"
else:
    neg_kind_desc = negative_kind

items_ranked_selection_method = (
    f"{selection_method}-"
    f"{neg_desc}-{neg_kind_desc}-"
    f"{max_desc}"
    f"u{num_users_to_sample}_"
    f"pp{positive_threshold}-{positive_cap}"
)

SAVE_DIR = (
    Path(BASE_DIR)
    / dataset
    / first_stage_model_name
    / items_ranked_selection_method
    / f"seed={seed}"
    / "data"
)
print(f"SAVE_DIR: {SAVE_DIR}")

In [ ]:
SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
all_test_data = data.test
all_test_users = data.test_users
all_item_ids = np.arange(data.n_item, dtype=int)

In [9]:
sampled_test_users = rng.choice(all_test_users, size=num_users_to_sample,
                                replace=False).tolist()

In [10]:
print(f"Sampled {len(sampled_test_users)} users for testing out of {len(all_test_users)} total test users.")

Sampled 10000 users for testing out of 10000 total test users.


In [ ]:
users_test_data = {}
users_test_data_len = {}
users_items_test_data_filtered = {}
for user_id in sampled_test_users:
    user_test_data = all_test_data[data.test[:, 0] == user_id][:, 1::]
    users_test_data[user_id] = user_test_data
    users_test_data_len[user_id] = user_test_data.shape[0]
    users_items_test_data_filtered[user_id] = user_test_data[
        user_test_data[:, 1] >= 4, 0
    ]

In [ ]:
# Get statistics about the number of test items per user
users_test_data_len_df = pd.DataFrame(list(users_test_data_len.items()), columns=["user_id", "test_data_len"])
users_test_data_len_df.head()

In [ ]:
users_test_data_len_df["test_data_len"].describe(percentiles=[0.25, 0.5, 0.75,
                                                              0.8, 0.85, 0.95])

In [ ]:
candidate_items_per_user = {}

for user_id in sampled_test_users:
    user_train_val_item_ids = np.array(data.train_val[data.train_val.user == user_id].item)
    user_test_items_full = np.unique(
        all_test_data[all_test_data[:, 0] == user_id][:, 1].astype(int)
    )

    # Select positives
    if len(user_test_items_full) <= positive_threshold:
        selected_positive_items = user_test_items_full.copy()
    else:
        selected_positive_items = rng.choice(
            user_test_items_full,
            size=positive_cap,
            replace=False
        )

    # Select negatives
    if negative_sampling_mode == "fixed":
        num_neg_to_sample = num_negatives
    else:
        num_neg_to_sample = int(negatives_per_positive * len(selected_positive_items))
        num_neg_to_sample = max(min_num_negatives, num_neg_to_sample)
        num_neg_to_sample = min(max_num_negatives, num_neg_to_sample)

    num_neg_to_sample = (min(num_neg_to_sample, max(0, max_size - len(selected_positive_items)))
                         if max_size is not None else num_neg_to_sample)

    negative_pool = np.setdiff1d(
        np.setdiff1d(all_item_ids, user_train_val_item_ids),
        user_test_items_full
    )

    if len(negative_pool) < num_neg_to_sample:
        num_neg_to_sample = len(negative_pool)

    if negative_kind == "random":
        sampled_negatives = rng.choice(negative_pool, size=num_neg_to_sample, replace=False)

    elif negative_kind in ["hard_es", "mixed_es"]:
        user_id_tensor = torch.tensor(user_id).to(device)

        item_means, item_stds = first_stage_model.predict_items_mean_and_var_for_user(user_id_tensor)

        if isinstance(item_means, torch.Tensor):
            item_means_np = item_means.detach().cpu().numpy()
        else:
            item_means_np = np.asarray(item_means)

        neg_es_scores = item_means_np[negative_pool]

        if negative_kind == "hard_es":
            hard_pool_size = min(len(negative_pool), hard_pool_factor * num_neg_to_sample)
            hard_top_indices = np.argsort(-neg_es_scores)[:hard_pool_size]
            hard_pool = negative_pool[hard_top_indices]

            sampled_negatives = rng.choice(hard_pool, size=num_neg_to_sample, replace=False)
        else:  # mixed_es
            num_hard = int(round(hard_negative_fraction * num_neg_to_sample))
            num_random = num_neg_to_sample - num_hard

            hard_pool_size = min(len(negative_pool), max(num_hard, hard_pool_factor * num_hard))

            hard_top_indices = np.argsort(-neg_es_scores)[:hard_pool_size]
            hard_pool = negative_pool[hard_top_indices]

            num_hard = min(num_hard, len(hard_pool))
            sampled_hard = rng.choice(hard_pool, size=num_hard, replace=False) if num_hard > 0 \
                else np.array([], dtype=int)

            remaining_negative_pool = np.setdiff1d(negative_pool, sampled_hard)

            num_random = min(num_random, len(remaining_negative_pool))
            sampled_random = rng.choice(remaining_negative_pool, size=num_random,
                                        replace=False) if num_random > 0 \
                else np.array([], dtype=int)

            sampled_negatives = np.concatenate([sampled_hard, sampled_random])
    else:
        raise ValueError(f"Unknown negative_kind: {negative_kind}")

    # Creating the final candidate set
    candidate_items = np.concatenate(
        [selected_positive_items, sampled_negatives]
    )
    rng.shuffle(candidate_items)

    candidate_items_per_user[user_id] = candidate_items.tolist()

In [ ]:
candidate_lengths = [len(v) for v in candidate_items_per_user.values()]
print("Candidate set size stats:")
print(pd.Series(candidate_lengths).describe())

example_user = sampled_test_users[0]
print("Example user:", example_user)
print("Num candidate items:", len(candidate_items_per_user[example_user]))

In [ ]:
with open(SAVE_DIR / "sampled_test_users.pkl", "wb") as f:
    pickle.dump(sampled_test_users, f)

with open(SAVE_DIR / "candidate_items_per_user.pkl", "wb") as f:
    pickle.dump(candidate_items_per_user, f)